In [1]:
import re
import os
import asyncio
import time as time_module
import random
import traceback
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone, date, timedelta, time
from calendar import monthrange
from functools import reduce
from io import StringIO, BytesIO

# Third-party imports
import pandas as pd
import requests
import urllib3
import matplotlib.pyplot as plt
import nest_asyncio
from bs4 import BeautifulSoup
# from pdfminer.high_level import extract_text
from dateutil import parser as date_parser

# Web automation imports
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Playwright imports
from playwright.async_api import async_playwright
from playwright.sync_api import sync_playwright

# Disable warnings and apply nest_asyncio
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
nest_asyncio.apply()

In [2]:
def adjust_hongkong(exdate):
    if exdate is None:
        exdate = date.today()
    elif isinstance(exdate, str):
        exdate = datetime.strptime(exdate, "%Y-%m-%d").date()
    elif isinstance(exdate, datetime):
        exdate = exdate.date()

    weekday = exdate.weekday()
    if weekday == 6:    # Sunday → saturday
        exdate -= timedelta(days=1)
    return exdate

In [3]:
def adjust_exdate(exdate):
    """If exdate is Sat/Sun, shift to Fri/Mon. Accepts str, datetime, or date."""
    if exdate is None:
        exdate = date.today()
    elif isinstance(exdate, str):
        exdate = datetime.strptime(exdate, "%Y-%m-%d").date()
    elif isinstance(exdate, datetime):
        exdate = exdate.date()
    return exdate

In [4]:
def get_last_available_date(exdate, check_func):
    """
    exdate: string 'YYYY-MM-DD' or date
    check_func: function that returns True if data exists for that date
    """
    # exdate = adjust_sun_sat_exdate(exdate)
    exdate = adjust_exdate(exdate)

    # Keep going backwards until data exists
    while True:
        if check_func(exdate):
            return exdate
        exdate -= timedelta(days=1)

In [5]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/123.0 Safari/537.36"
}
def scrape_hongkong(exdate: str):
    def hongkong_has_data(check_date: date):
        """Return True if HKAB has data for the given date."""
        # check_date is a date object here
        date_str = check_date.strftime("%Y-%m-%d")
        target = adjust_hongkong(date_str)  # whatever format HK API needs
        url = f"https://www.hkab.org.hk/api/member/public/getExrate/{target}"

        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code != 200:
                return False

            data = r.json()
            if not data:
                return False

            # Must contain valid fields
            return ("USDSelling" in data) and ("USDBuyingTT" in data)
        except Exception:
            return False

    # Step 1 — find the actual available date (returns a date object)
    real_date = get_last_available_date(exdate, hongkong_has_data)

    # Step 2 — format for API using the REAL date
    real_date_str = real_date.strftime("%Y-%m-%d")
    target = adjust_hongkong(real_date_str)
    HIST_URL = f"https://www.hkab.org.hk/api/member/public/getExrate/{target}"

    # Step 3 — fetch actual data
    r = requests.get(HIST_URL, headers=HEADERS, timeout=15)
    r.raise_for_status()
    data = r.json()

    sell = float(data["USDSelling"])
    buy  = float(data["USDBuyingTT"])
    mid  = (sell + buy) / 2.0

    # Step 4 — return full record
    return [{
        "country": "Hong Kong",
        "value": mid,
        "unit": "HKD",
        "website": "https://www.hkab.org.hk/en/rates/exchange-rates",
        # use *actual* last available date, not the requested one
        "date_of_page": exdate,
        "Source": "The Hong Kong Association of Banks",
        "Status": "buy, sell",
    }]

In [6]:
def scrape_lao(exdate: str):
    url = "https://www.bol.gov.la/en/ExchangRate"

    def lao_has_data(check_date: date) -> bool:
        """Return True if BOL has USD data for this date."""
        query_date_str = check_date.strftime("%d-%m-%Y")

        payload = {
            "date": query_date_str,
            "search": "Search"
        }

        try:
            r = requests.post(
                url,
                headers=HEADERS,
                data=payload,
                timeout=15,
                verify=False,  # their SSL cert chain is broken
            )
            if r.status_code != 200:
                return False

            soup = BeautifulSoup(r.text, "html.parser")
            table = soup.find("table")
            if table is None:
                return False

            # Check if there's a USD row
            for tr in table.find_all("tr"):
                tds = tr.find_all("td")
                if len(tds) < 6:
                    continue
                code = tds[3].get_text(strip=True).upper()
                if code == "USD":
                    return True

            return False
        except Exception:
            return False

    # 1) Find the last date that actually has data
    real_date = get_last_available_date(exdate, lao_has_data)
    query_date_str = real_date.strftime("%d-%m-%Y")

    # 2) Fetch page for that actual date
    payload = {
        "date": query_date_str,
        "search": "Search"
    }

    r = requests.post(
        url,
        headers=HEADERS,
        data=payload,
        timeout=15,
        verify=False,
    )
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # Extract displayed date from page (look for "Date: DD-MM-YYYY")
    date_text = None
    for elem in soup.find_all(string=True):
        if "Date:" in elem:
            m = re.search(r"(\d{2}-\d{2}-\d{4})", elem)
            if m:
                date_text = m.group(1)
                break
    if not date_text:
        raise RuntimeError("No 'Date: DD-MM-YYYY' found on page. The page may be empty or the date is invalid.")

    page_date = datetime.strptime(date_text, "%d-%m-%Y").date()
    print(f"✅ Lao page date: {page_date}, requested: {exdate}, effective: {real_date}")

    # Find the table
    table = soup.find("table")
    if table is None:
        raise RuntimeError("No table found on BOL exchange rate page")

    usd_buy = usd_sell = None

    def parse_lao_number(s: str) -> float:
        s = s.strip().replace(".", "").replace(",", ".")  # dot=thousands, comma=decimal
        try:
            return float(s)
        except Exception as e:
            raise ValueError(f"Cannot parse '{s}'") from e

    # Each row: No | Countries | Foreign Currencies | Currency Code | Buy Rates | Sell Rates
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 6:
            continue

        code = tds[3].get_text(strip=True).upper()

        if code == "USD":
            buy_str = tds[4].get_text(strip=True)
            sell_str = tds[5].get_text(strip=True)

            usd_buy = parse_lao_number(buy_str)
            usd_sell = parse_lao_number(sell_str)
            break

    if usd_buy is None or usd_sell is None:
        codes = [tds[3].get_text(strip=True) for tr in table.find_all("tr")
                 for tds in [tr.find_all("td")] if len(tds) >= 4]
        raise RuntimeError(f"USD row not found. Available currency codes: {codes}")

    mid = (usd_buy + usd_sell) / 2.0

    return [{
        "country": "Lao",
        "unit": "LAK",
        "buy": usd_buy,
        "sell": usd_sell,
        "value": mid,                           # what you asked for
        "date_of_page": exdate,     # actual data date
        "website": url,
        "Source": "Bank of the lao P.D.R",
        "Status": "buy, sell"
    }]

In [7]:
def scrape_myanmar(exdate=None):
    base_url = "https://forex.cbm.gov.mm/api/history"

    # If no date provided, start from "today" (UTC) as YYYY-MM-DD string
    if exdate is None:
        start_date_str = datetime.utcnow().strftime("%Y-%m-%d")
    else:
        start_date_str = exdate  # expected "YYYY-MM-DD"

    def myanmar_has_data(check_date: date) -> bool:
        """
        Returns True if CBM API has USD data for the given date.
        API format is dd-mm-yyyy.
        """
        dt_for_api = check_date.strftime("%d-%m-%Y")
        url = f"{base_url}/{dt_for_api}"

        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            if r.status_code != 200:
                return False

            data = r.json()

            # According to your comment, "no data" => [] (empty list)
            if isinstance(data, list):
                return False

            # Should be a dict with "rates" and "USD"
            rates = data.get("rates", {})
            return "USD" in rates
        except Exception:
            return False

    # 1) Find the last date that actually has data
    real_date = get_last_available_date(start_date_str, myanmar_has_data)
    dt_for_api = real_date.strftime("%d-%m-%Y")
    dt_for_page = real_date.strftime("%Y-%m-%d")

    # 2) Fetch the actual data for that date
    url = f"{base_url}/{dt_for_api}"
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.raise_for_status()

    data = r.json()

    # Handle unexpected formats defensively
    if isinstance(data, list):
        if not data:
            raise ValueError(f"No Myanmar CBM rate data for date {dt_for_page}")
        else:
            raise ValueError(f"Unexpected JSON format from CBM API for date {dt_for_page}: list with elements")

    rates = data.get("rates", {})
    if "USD" not in rates:
        raise ValueError(f"USD rate not found in CBM response for date {dt_for_page}")

    usd_rate_str = rates["USD"]
    usd_rate = float(usd_rate_str)

    return [{
        "country": "Myanmar",
        "value": usd_rate,                        # e.g. 2100.0
        "unit": "MMK",                            # 1 USD = X MMK
        "requested_date": exdate or start_date_str,  # what you asked for
        "date_of_page": exdate,              # actual data date (after holiday adjustment)
        "website": "https://forex.cbm.gov.mm/index.php/fxrate/history",
        "Source": "Central Bank of Myanmar",
        "Status": "only one"
    }]

In [8]:
def _is_number_line(s: str) -> bool:
    s = s.strip()
    if not s:
        return False
    # digits, optional commas, optional decimal part – e.g. 280.95, 1,234.50
    return bool(re.fullmatch(r"\d[\d,]*\.?\d*", s))

In [9]:
def scrape_pakistan(exdate=None):
    base_url = "https://www.nbp.com.pk/RateSheetFiles"

    # --- 1. Choose starting date (requested or today) as YYYY-MM-DD string ---
    if exdate is None:
        start_date_str = datetime.utcnow().strftime("%Y-%m-%d")
    else:
        start_date_str = exdate  # expected 'YYYY-MM-DD'

    # --- 2. Define "has data" checker for a given date ---
    def pakistan_has_data(check_date: date) -> bool:
        dt_for_pdf = check_date.strftime("%d-%m-%Y")  # dd-mm-yyyy
        url = f"{base_url}/NBP-RateSheet-{dt_for_pdf}.pdf"

        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code != 200:
                return False

            pdf_text = extract_text(BytesIO(r.content))
            # Simple sanity check: file exists and mentions US DOLLAR
            return "US DOLLAR" in pdf_text.upper()
        except Exception:
            return False

    # --- 3. Find last date where the PDF exists and has USD data ---
    real_date = get_last_available_date(start_date_str, pakistan_has_data)
    dt_for_pdf  = real_date.strftime("%d-%m-%Y")   # for PDF filename
    dt_for_page = real_date.strftime("%Y-%m-%d")   # stored in Excel / output

    url = f"{base_url}/NBP-RateSheet-{dt_for_pdf}.pdf"

    # --- 4. Fetch PDF for the actual (holiday-adjusted) date ---
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()

    pdf_text = extract_text(BytesIO(r.content))
    lines = [ln.strip() for ln in pdf_text.splitlines() if ln.strip()]

    # --- 5. Find 'US DOLLAR' line index ---
    usd_idx = None
    for i, ln in enumerate(lines):
        if "US DOLLAR" in ln.upper():
            usd_idx = i
            break

    if usd_idx is None:
        raise ValueError(f"'US DOLLAR' not found in NBP PDF for {dt_for_page}")

    # --- 6. Scan forward for numeric lines and collect them ---
    numeric_vals = []
    for ln in lines[usd_idx + 1 :]:
        if _is_number_line(ln):
            val = float(ln.replace(",", ""))
            numeric_vals.append(val)
            if len(numeric_vals) > 38:
                break

    if len(numeric_vals) < 39:
        raise ValueError(
            f"Not enough numeric rates found after USD row. "
            f"Found {len(numeric_vals)} numbers: {numeric_vals}"
        )

    # Your existing logic: buy = numeric_vals[38]
    buy = numeric_vals[38]
    sell = numeric_vals[25]
    mid = (buy + sell) / 2.0

    return [{
        "country": "Pakistan",
        "value": mid,                      # update
        "unit": "PKR",                     # 1 USD = X PKR
        "date_of_page": exdate,       # actual date of the PDF / rate
        "website": url,
        "Source": "National Bank of Pakistan",
        "Status": "buy",
    }]

In [10]:
def scrape_vietnam(exdate=None):
    """
    Vietnam – SBV reference rate.
    Uses get_last_available_date to walk backwards until a day with data is found.
    """

    # 1) Decide starting date (requested or today)
    if exdate is None:
        start_date_str = datetime.utcnow().strftime("%Y-%m-%d")
    else:
        start_date_str = exdate  # expected 'YYYY-MM-DD'

    BASE_URL = (
        "https://sbv.gov.vn/o/headless-delivery/v1.0/"
        "content-structures/3450514/structured-contents"
    )

    HEADER = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/123.0 Safari/537.36",
        "Accept": "application/json"
    }

    def vietnam_has_data(check_date: date) -> bool:
        """Return True if SBV has any USD rate item for this date."""
        d_str = check_date.strftime("%Y-%m-%d")
        end_str = (check_date + timedelta(days=1)).strftime("%Y-%m-%d")

        url = (
            f"{BASE_URL}"
            f"?filter=datePublished%20ge%20{d_str}T00%3A00%3A00.000Z"
            f"%20and%20datePublished%20le%20{end_str}T00%3A00%3A00.000Z"
        )

        try:
            r = requests.get(url, headers=HEADER, timeout=15)
            if r.status_code != 200:
                return False

            data = r.json()
            items = data.get("items", [])
            if not items:
                return False

            # Optional: check USD exists in at least one item
            for item in items:
                content_fields = item.get("contentFields", [])
                for field in content_fields:
                    if field.get("name") == "tyGiaThamKhaos" and field.get("repeatable"):
                        nested = field.get("nestedContentFields", [])
                        for sub in nested:
                            if (
                                sub.get("name") == "ngoaiTe"
                                and sub.get("contentFieldValue", {}).get("data", "").startswith("USD")
                            ):
                                return True
            return False
        except Exception:
            return False

    # 2) Find the last date with valid USD data
    real_date = get_last_available_date(start_date_str, vietnam_has_data)
    d_str = real_date.strftime("%Y-%m-%d")
    end_date = (real_date + timedelta(days=1)).strftime("%Y-%m-%d")

    url = (
        f"{BASE_URL}"
        f"?filter=datePublished%20ge%20{d_str}T00%3A00%3A00.000Z"
        f"%20and%20datePublished%20le%20{end_date}T00%3A00%3A00.000Z"
    )

    # 3) Fetch actual data for that date
    try:
        r = requests.get(url, headers=HEADER, timeout=15)
        r.raise_for_status()
    except requests.RequestException as e:
        raise RuntimeError(f"HTTP request failed for Vietnam SBV: {e}")

    try:
        data = r.json()
    except ValueError as e:
        raise RuntimeError(f"Failed to parse JSON from SBV: {e}")

    items = data.get("items", [])
    if not items:
        raise RuntimeError(f"No exchange rate items found for Vietnam on {d_str}.")

    item = items[0]
    content_fields = item.get("contentFields", [])

    # Parse reference date & rates
    ref_date = None
    rates = []

    for field in content_fields:
        name = field.get("name")

        if name == "ngayApDung":
            val = field.get("contentFieldValue", {}).get("data")
            if val:
                try:
                    dt = date_parser.isoparse(val)
                    ref_date = dt.astimezone(timezone(timedelta(hours=7))).date()  # ICT
                except Exception:
                    pass  # fallback later

        if name == "tyGiaThamKhaos" and field.get("repeatable"):
            nested = field.get("nestedContentFields", [])
            curr = {}
            for sub in nested:
                n = sub.get("name")
                v = sub.get("contentFieldValue", {}).get("data")
                if n == "ngoaiTe" and v:
                    parts = v.split("-", 1)
                    curr["currency"] = parts[0].strip()
                    curr["name"] = parts[1].strip() if len(parts) > 1 else v
                elif n == "mua":
                    try:
                        curr["buy"] = float(v) if v else 0.0
                    except Exception:
                        curr["buy"] = 0.0
                elif n == "ban":
                    try:
                        curr["sell"] = float(v) if v else 0.0
                    except Exception:
                        curr["sell"] = 0.0
            if curr.get("currency"):
                rates.append(curr)

    if not rates:
        raise RuntimeError("No currency rates extracted from Vietnam SBV contentFields.")

    usd = next((r for r in rates if r["currency"] == "USD"), None)
    if usd is None:
        raise RuntimeError("USD rate not found in Vietnam SBV data.")

    mid = (usd["buy"] + usd["sell"]) / 2.0

    # Prefer ref_date if parsed, otherwise fall back to real_date
    effective_date = (ref_date or real_date).strftime("%Y-%m-%d")

    return [{
        "country": "Vietnam",
        "value": mid,
        "unit": "VND",
        "date_of_page": exdate,             # real SBV rate date
        "website": "https://sbv.gov.vn/vi/t%E1%BB%B7-gi%C3%A1-tham-kh%E1%BA%A3o-gi%E1%BB%AFa-%C4%91%E1%BB%93ng-vi%E1%BB%87t-nam-v%C3%A0-c%C3%A1c-lo%E1%BA%A1i-ngo%E1%BA%A1i-t%E1%BB%87-t%E1%BA%A1i-c%E1%BB%A5c-qu%E1%BA%A3n-l%C3%BD-ngo%E1%BA%A1i-h%E1%BB%91i",
        "Source": "SBV",
        "Status": "buy, sell",
    }]

In [ ]:
import asyncio

async def main():
    TARGET_DATE = "2026-3-10"

    # Store function references directly, not lambdas that create coroutines
    scraper_functions = [
        # scrap_IMF, # Not async
        # scrape_cambodia, # Not async
        # scrape_hongkong, # Not async
        # # scrape_indonesia, # Not async
        # indonesia_scrape,
        # scrape_lao, # Not async
        # scrape_myanmar, # Not async
        scrape_pakistan, # Not async
        # scrape_vietnam, # Not async
        # scrape_sri_lanka, # Not async
        # scrape_bangladesh, # This is async
    #     turkey_scrape, # This is async
    #     egypt_scrape, # This is async
    ]

    rows = []
    scraped_at = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    for func in scraper_functions:
        try:
            if asyncio.iscoroutinefunction(func):
                # If it's an async function, await its call with TARGET_DATE
                result_list = await func(TARGET_DATE)
            else:
                # If it's a sync function, call it directly with TARGET_DATE
                result_list = func(TARGET_DATE)

            if not result_list:
                print(f"⚠️ {func.__name__} returned no data")
                continue

            for item in result_list:
                item["date_of_scrape"] = scraped_at
                rows.append(item)
                print(f"OK: {item['country']} → {item['value']} ({item['unit']}) @ {item['date_of_page']}")
        except Exception as e:
            print(f"❌ ERROR in {func.__name__}: {e}")

    if not rows:
        print("❌ No data scraped. Exiting.")
        return

    # Build DataFrame
    df = pd.DataFrame(
        rows,
        columns=["country", "value", "unit", "date_of_page", "date_of_scrape", "Status", "website", "Source"],
    )

# ========================================================#
#    this is the file output of data that we scrape       #
#       you can change name aas you see fit               #
# ========================================================#
    # output_file = "/content/drive/MyDrive/Dataset/Exchange_rate/usd_exchange_rate_with_another_national_banks.xlsx"
    # # output_file = "/content/drive/MyDrive/Dataset/Exchange_rate/usd_exchange_rate_with_other_countries_national_banks.xlsx"

    # # Save or append
    # if os.path.exists(output_file):
    #     old_df = pd.read_excel(output_file)
    #     combined = pd.concat([old_df, df], ignore_index=True)
    #     # Deduplicate by country + date_of_page (keep latest scrape)
    #     combined.drop_duplicates(
    #         subset=["country", "date_of_page"], keep="last", inplace=True
    #     )
    #     combined.to_excel(output_file, index=False)
    #     print(f"\n✅ Updated {output_file} with {len(df)} new rows (total: {len(combined)}).")
    # else:
    #     df.to_excel(output_file, index=False)
    #     print(f"\n✅ Created new file: {output_file} with {len(df)} rows.")

    # print("\n🎉 Done.")


if __name__ == "__main__":
    asyncio.run(main())

C:\Users\Admin\AppData\Local\Temp\ipykernel_7516\203717584.py:28: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  if asyncio.iscoroutinefunction(func):


OK: Hong Kong → 782.075 (HKD) @ 2026-3-10
✅ Lao page date: 2026-03-10, requested: 2026-3-10, effective: 2026-03-10
OK: Lao → 21429.5 (LAK) @ 2026-3-10
OK: Myanmar → 2100.0 (MMK) @ 2026-3-10


KeyboardInterrupt: 